# FD-CMKD from scratch

The distillation runs reported in the thesis are **warm-started**: they load the
converged baseline checkpoint and fine-tune it with the transfer loss. That design
answers *does the teacher improve an already-trained student?* -- and, under a
matched control, the answer was no.

It leaves a different question open: *does the teacher help when present from the
start?* A teacher seen from random initialization can shape the representation as it
forms, rather than nudge a model already settled in a minimum. This notebook trains
the full FD-CMKD student **from scratch** so that question can be answered too.

**What makes the comparison fair.** Every training hyperparameter in
`from_scratch_fd_cmkd.yaml` is copied verbatim from the baseline `sign.yaml`, and the
config carries no `load_model`. The only difference between this run and the CTC-only
baseline (39.54 dev / 41.53 test) is the distillation loss. To read a result as
evidence rather than anecdote, give this run the same learning-rate search the
baseline had (the Optuna cell), and fix a decision margin before you look.

**The warmup ramp.** From scratch, the projection and the shared classifiers start
random, so their gradient is noise until CTC pulls the encoder out of blank collapse
(the first couple of epochs). `distill_warmup_steps` ramps the distillation weight in
linearly over that window; without it, a from-scratch run can be destabilised early
and you would not know whether the teacher or the initialization was at fault.

In [ ]:
# Run from code/distillation/ no matter where the kernel was launched.
import os
if not os.path.exists('fd_cmkd.py'):
    _cwd = os.getcwd()
    for _k in range(0, 6):
        _base = os.path.abspath(os.path.join(_cwd, *(['..'] * _k)))
        _cand = os.path.join(_base, 'code', 'distillation')
        if os.path.exists(os.path.join(_cand, 'fd_cmkd.py')):
            os.chdir(_cand); break
        if os.path.exists(os.path.join(_base, 'fd_cmkd.py')):
            os.chdir(_base); break
print('working dir:', os.getcwd())

# Where this run writes -- same convention as the other notebooks.
# Output goes to runs/<RUN_NAME>/; dataset/checkpoints/ is never touched.
import os

RUN_NAME  = 'forward_scratch'
OVERWRITE = False        # False: version to _002, _003, ... instead of deleting
DO_TRAIN  = False        # True: train from scratch (up to 1000 epochs, early-stopped)

RUN_DIR = os.path.join('../../runs', RUN_NAME)
if os.path.exists(RUN_DIR) and not OVERWRITE:
    _n = 2
    while os.path.exists('%s_%03d' % (RUN_DIR, _n)):
        _n += 1
    RUN_DIR = '%s_%03d' % (RUN_DIR, _n)
print('this run writes to:', os.path.normpath(os.path.abspath(RUN_DIR)))

# This notebook evaluates only its own run.
EVAL_CKPT = 'mine'

In [ ]:
import sys, yaml

SIGNFORMER_DIR = os.path.abspath('../signformer')   # for `import main.*`
NB_DIR = os.path.abspath('.')                        # for `import fd_cmkd*`
for _p in (SIGNFORMER_DIR, NB_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import main.training as mt
from fd_cmkd_trainer import train_fd_cmkd
from fd_cmkd import load_teacher_feats
print('imports OK')

In [ ]:
# Sanity checks: data and teacher features present, config genuinely from-scratch.
BASE_CFG = 'from_scratch_fd_cmkd.yaml'
assert os.path.exists(BASE_CFG), 'missing config: ' + BASE_CFG

cfg = yaml.safe_load(open(BASE_CFG))
dc = cfg['training']['distillation']

i3d = os.path.join(cfg['data']['data_path'], cfg['data']['train'])
assert os.path.exists(i3d), 'missing I3D features: ' + i3d

tfp = os.path.join(dc['teacher_feats_dir'], 'train.pkl')
assert os.path.exists(tfp), 'missing teacher features: ' + tfp
_t = load_teacher_feats(tfp)

assert 'load_model' not in cfg['training'], 'a from-scratch run must not warm-start'
print('i3d train features : OK')
print('teacher features   : %d videos, dim %d' % (len(_t), next(iter(_t.values())).shape[1]))
print('warm-start         : none (trains from random init)')
print('distill warmup     : %d steps' % dc['distill_warmup_steps'])

In [ ]:
# Point the config at RUN_DIR, honour OVERWRITE, write the resolved config there.
# The committed from_scratch_fd_cmkd.yaml is never modified.
# RUN_DIR is a container. The trainer writes into RUN_DIR/final, which it must
# be free to create fresh: make_model_dir refuses a directory that already
# exists. Writing the config into RUN_DIR (not into MODEL_DIR) keeps them apart.
MODEL_DIR = os.path.join(RUN_DIR, 'final')
cfg['training']['model_dir'] = MODEL_DIR
cfg['training']['overwrite'] = OVERWRITE

os.makedirs(RUN_DIR, exist_ok=True)
ACTIVE_CFG = os.path.join(RUN_DIR, 'from_scratch_fd_cmkd.yaml')
with open(ACTIVE_CFG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

tr = cfg['training']
print('active config ->', ACTIVE_CFG)
print('lr=%.3e  wd=%.3e  batch=%d  patience=%d' % (tr['learning_rate'], tr['weight_decay'], tr['batch_size'], tr['early_stopping_patience']))
print('lambda_feat=%.3f  lambda_align=%.3f  w_high=%.3f  warmup=%d' % (dc['lambda_feat'], dc['lambda_align'], dc['high_w'], dc['distill_warmup_steps']))

## Optional: matched learning-rate search

The baseline was selected over a small Optuna search. For the comparison to be fair,
the from-scratch distilled arm deserves a search of the **same budget** over the
**same learning-rate range**. Turn it on to run it; each trial writes to a temporary
directory under `RUN_DIR`, and the best learning rate is kept.

Leave it off (`USE_OPTUNA = False`) to train a single run at the baseline learning
rate -- a quick check that the pipeline works, but not, on its own, evidence.

In [ ]:
import copy, shutil, gc

USE_OPTUNA    = False   # True re-runs the matched search (hours)
OPTUNA_TRIALS = 6       # match the baseline budget
LR_RANGE      = (1.5e-4, 6e-4)   # a neighbourhood of the baseline 3.12e-4

if USE_OPTUNA and DO_TRAIN:
    import torch
    try:
        import optuna
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
        import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    _log = []
    def _objective(trial):
        lr = trial.suggest_float('learning_rate', LR_RANGE[0], LR_RANGE[1], log=True)
        ct = copy.deepcopy(cfg)
        ct['training']['learning_rate'] = float(lr)
        td = os.path.join(RUN_DIR, 'optuna', 'trial_%02d' % trial.number)
        os.makedirs(td, exist_ok=True)
        ct['training']['model_dir'] = td
        ct['training']['overwrite'] = True
        cf = os.path.join(ct['training']['model_dir'], 'cfg.yaml')
        with open(cf, 'w') as f:
            yaml.safe_dump(ct, f, sort_keys=False)
        train_fd_cmkd(cf)
        wers = []
        vf = os.path.join(ct['training']['model_dir'], 'validations.txt')
        if os.path.exists(vf):
            for line in open(vf):
                toks = line.replace(':', ' ').split()
                if 'WER' in toks:
                    try:
                        wers.append(float(toks[toks.index('WER') + 1]))
                    except (ValueError, IndexError):
                        pass
        best = min(wers) if wers else 100.0
        _log.append((trial.number, lr, best))
        print('  trial %d: lr=%.3e -> dev WER %.2f' % (trial.number, lr, best))
        shutil.rmtree(td, ignore_errors=True)
        gc.collect(); torch.cuda.empty_cache()
        return best

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_objective, n_trials=OPTUNA_TRIALS)
    print('best lr:', study.best_params, '-> dev WER', round(study.best_value, 2))
    cfg['training']['learning_rate'] = float(study.best_params['learning_rate'])
    with open(ACTIVE_CFG, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print('ACTIVE_CFG updated with the searched learning rate.')
else:
    print('Optuna search disabled (or DO_TRAIN=False) -- using the config lr.')

In [ ]:
# Train from scratch. This is the long cell.
if DO_TRAIN:
    train_fd_cmkd(ACTIVE_CFG)
else:
    print('DO_TRAIN = False -> skipping training. Set it True to run from scratch.')

In [ ]:
# Development-WER curve against the baseline greedy 39.54.
import matplotlib.pyplot as plt

val_file = os.path.join(RUN_DIR, 'final', 'validations.txt')
if not os.path.exists(val_file):
    print('no validations.txt yet:', val_file)
else:
    steps, wer = [], []
    for line in open(val_file):
        toks = line.replace(':', ' ').split()
        if 'Steps' in toks and 'WER' in toks:
            try:
                steps.append(int(toks[toks.index('Steps') + 1]))
                wer.append(float(toks[toks.index('WER') + 1]))
            except (ValueError, IndexError):
                pass
    plt.figure(figsize=(7, 4))
    plt.plot(steps, wer, marker='o', ms=3, label='from-scratch FD-CMKD')
    plt.axhline(39.54, ls='--', c='gray', label='baseline (greedy) 39.54')
    plt.xlabel('step'); plt.ylabel('dev WER (%)'); plt.legend()
    plt.title('FD-CMKD from scratch vs baseline')
    plt.savefig(os.path.join(RUN_DIR, 'final', 'training_curves.png'), dpi=110)
    plt.show()
    if wer:
        print('best dev WER reached: %.2f' % min(wer))

In [ ]:
# Evaluate the best-on-dev checkpoint and lay the numbers next to the baseline.
best_ckpt = os.path.join(RUN_DIR, 'final', 'best.ckpt')
if not os.path.exists(best_ckpt):
    print('no best.ckpt yet -- train first (DO_TRAIN = True).')
else:
    mt.test(ACTIVE_CFG, ckpt=best_ckpt, output_path=os.path.join(RUN_DIR, 'final', 'test_output'))
    print()
    print('Baseline, from scratch, CTC only (sign.yaml):')
    print('   dev  39.54 greedy / 39.35 beam-4      test  41.53')
    print('This run, from scratch, CTC + FD-CMKD:')
    print('   see the [DEV] and [TEST] WER printed above')
    print()
    print('Reading the result:')
    print(' - below baseline beyond the margin: warm-start was hiding a real transfer.')
    print(' - within the margin: the last alternative explanation is ruled out.')
    print(' - above baseline: a weak teacher shaping the representation transfers its confusion.')